# EDA and label audit

Explore sanitized review length, common automotive slang, and canonical label balance. The checked-in sample is only a smoke-test fixture; use an approved, de-identified corpus for meaningful analysis.

Imports and project paths

In [7]:
from pathlib import Path
import zipfile
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

RANDOM_STATE = 42

PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data"
DATA_DIR.mkdir(exist_ok=True)

ZIP_PATH = DATA_DIR / "nhtsa_complaints_2020_2024.zip"

print("Project root:", PROJECT_ROOT)
print("Data directory:", DATA_DIR)

Project root: d:\ML_Prep\automotive-ner-pipeline
Data directory: d:\ML_Prep\automotive-ner-pipeline\data


Download the dataset

In [8]:
DATA_URL = (
    "https://static.nhtsa.gov/odi/ffdd/cmpl/"
    "COMPLAINTS_RECEIVED_2020-2024.zip"
)

if not ZIP_PATH.exists():
    response = requests.get(DATA_URL, stream=True, timeout=60)
    response.raise_for_status()

    with open(ZIP_PATH, "wb") as file:
        for chunk in response.iter_content(chunk_size=1024 * 1024):
            if chunk:
                file.write(chunk)

print(f"Downloaded: {ZIP_PATH}")
print(f"Size: {ZIP_PATH.stat().st_size / (1024**2):.2f} MB")

Downloaded: d:\ML_Prep\automotive-ner-pipeline\data\nhtsa_complaints_2020_2024.zip
Size: 72.36 MB


Inspect the archive

In [9]:
with zipfile.ZipFile(ZIP_PATH, "r") as archive:
    for info in archive.infolist():
        print(
            info.filename,
            f"{info.file_size / (1024**2):.2f} MB"
        )

COMPLAINTS_RECEIVED_2020-2024.txt 311.56 MB


Define the schema

In [10]:
columns = [
    "CMPLID", "ODINO", "MFR_NAME", "MAKETXT", "MODELTXT",
    "YEARTXT", "CRASH", "FAILDATE", "FIRE", "INJURED",
    "DEATHS", "COMPDESC", "CITY", "STATE", "VIN",
    "DATEA", "LDATE", "MILES", "OCCURENCES", "CDESCR",
    "CMPL_TYPE", "POLICE_RPT_YN", "PURCH_DT",
    "ORIG_OWNER_YN", "ANTI_BRAKES_YN", "CRUISE_CONT_YN",
    "NUM_CYLS", "DRIVE_TRAIN", "FUEL_SYS", "FUEL_TYPE",
    "TRANS_TYPE", "VEH_SPEED", "DOT", "TIRE_SIZE",
    "LOC_OF_TIRE", "TIRE_FAIL_TYPE", "ORIG_EQUIP_YN",
    "MANUF_DT", "SEAT_TYPE", "RESTRAINT_TYPE",
    "DEALER_NAME", "DEALER_TEL", "DEALER_CITY",
    "DEALER_STATE", "DEALER_ZIP", "PROD_TYPE",
    "REPAIRED_YN", "MEDICAL_ATTN", "VEHICLES_TOWED_YN",
    "STATE_OF_INCIDENT", "VEHICLE_OPERATOR"
]

assert len(columns) == 51

Load into Pandas

In [11]:
with zipfile.ZipFile(ZIP_PATH, "r") as archive:
    data_files = [
        name for name in archive.namelist()
        if name.lower().endswith((".txt", ".lst"))
    ]

    assert len(data_files) == 1, data_files

    with archive.open(data_files[0]) as file:
        df = pd.read_csv(
            file,
            sep="\t",
            header=None,
            names=columns,
            dtype="string",
            encoding="latin-1",
            keep_default_na=False,
            on_bad_lines="warn"
        )

print("Dataset shape:", df.shape)
display(df.head(3))

Dataset shape: (418806, 51)


,CMPLID,ODINO,MFR_NAME,MAKETXT,MODELTXT,YEARTXT,CRASH,FAILDATE,FIRE,INJURED,...,DEALER_TEL,DEALER_CITY,DEALER_STATE,DEALER_ZIP,PROD_TYPE,REPAIRED_YN,MEDICAL_ATTN,VEHICLES_TOWED_YN,STATE_OF_INCIDENT,VEHICLE_OPERATOR
0,1633421,11292384,Honda (American Honda Motor Co.),HONDA,ACCORD,2018,N,20191221,N,0,...,,,,,V,,N,N,,
1,1633422,11292384,Honda (American Honda Motor Co.),HONDA,ACCORD,2018,N,20191221,N,0,...,,,,,V,,N,N,,
2,1633423,11292384,Honda (American Honda Motor Co.),HONDA,ACCORD,2018,N,20191221,N,0,...,,,,,V,,N,N,,


Dataset structure

In [12]:
print("Rows:", len(df))
print("Columns:", len(df.columns))

df.info()

Rows: 418806
Columns: 51
<class 'pandas.DataFrame'>
RangeIndex: 418806 entries, 0 to 418805
Data columns (total 51 columns):
 #   Column             Non-Null Count   Dtype 
---  ------             --------------   ----- 
 0   CMPLID             418806 non-null  string
 1   ODINO              418806 non-null  string
 2   MFR_NAME           418806 non-null  string
 3   MAKETXT            418806 non-null  string
 4   MODELTXT           418806 non-null  string
 5   YEARTXT            418806 non-null  string
 6   CRASH              418806 non-null  string
 7   FAILDATE           418806 non-null  string
 8   FIRE               418806 non-null  string
 9   INJURED            418806 non-null  string
 10  DEATHS             418806 non-null  string
 11  COMPDESC           418806 non-null  string
 12  CITY               418806 non-null  string
 13  STATE              418806 non-null  string
 14  VIN                418806 non-null  string
 15  DATEA              418806 non-null  string
 16  LDATE 

Missing and empty values

In [13]:
empty_counts = (
    df.eq("")
      .sum()
      .sort_values(ascending=False)
)

missing_summary = pd.DataFrame({
    "empty_count": empty_counts,
    "empty_percentage": (
        empty_counts / len(df) * 100
    ).round(2)
})

display(missing_summary)

,empty_count,empty_percentage
STATE_OF_INCIDENT,418806,100.0
VEHICLE_OPERATOR,418806,100.0
FUEL_SYS,418515,99.93
TIRE_SIZE,418457,99.92
MANUF_DT,418217,99.86
RESTRAINT_TYPE,418190,99.85
SEAT_TYPE,418140,99.84
TIRE_FAIL_TYPE,418136,99.84
TRANS_TYPE,417196,99.62
ORIG_EQUIP_YN,417110,99.6


Check narrative quality

In [14]:
df["narrative_length"] = (
    df["CDESCR"].fillna("").str.len()
)

df["word_count"] = (
    df["CDESCR"]
    .fillna("")
    .str.split()
    .str.len()
)

print("Empty narratives:", (df["CDESCR"].str.strip() == "").sum())
print("Median narrative length:", df["narrative_length"].median())
print("Mean narrative length:", df["narrative_length"].mean())

display(
    df[["CDESCR", "narrative_length", "word_count"]]
    .sample(min(10, len(df)), random_state=RANDOM_STATE)
)

Empty narratives: 0
Median narrative length: 499.0
Mean narrative length: 591.8812481196544


,CDESCR,narrative_length,word_count
408052,There has been a promise to rectify this issue...,160,29
334286,Vehicle was taken to the RV dealer to have ann...,684,119
5884,TL* THE CONTACT OWNS A 2008 FORD EDGE. THE VEH...,1119,196
322366,I was on my way to work. All the warning light...,769,146
94216,TL* THE CONTACT OWNS A 2009 BUICK LUCERNE. THE...,534,90
299574,When beginning to drive forward after coming t...,858,157
313546,The contact owns a 2015 Mercedes-Benz E350. Th...,597,94
157089,1. ABS SYSTEM- ANTI-LOCK BRAKING SYSTEM SENSOR...,429,60
235085,The contact owns a 2014 Kia Sorento. The conta...,509,82
85098,TL* THE CONTACT OWNS A 2012 CHEVROLET CRUZE. T...,907,145
